# Exploratory Data Analysis & Statistical Insights
**Goal:** uncover business patterns using descriptive statistics, distributions, correlations, anomaly detection and formal hypothesis tests.

**How to use:** set `DATA_PATH` in the first code cell to your cleaned CSV (e.g. `clean_dataset.csv`) and adjust the `COL` mapping. If the file is not found, the notebook generates a synthetic retail-customer dataset so every cell still runs.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", "{:,.3f}".format)

# ---- CONFIG: point to your data and map the columns used in the hypotheses ----
DATA_PATH = "clean_dataset.csv"
COL = {
    "discount":  "discount_rate",     # numeric, 0-1
    "retained":  "retained",          # binary 0/1 (customer retained / repeat buyer)
    "spend":     "order_value",       # numeric revenue per customer/order
    "segment":   "segment",           # categorical (region, tier, channel...)
    "tenure":    "tenure_months",     # numeric
    "frequency": "orders_per_year",   # numeric
}

In [ ]:
import os
def make_synthetic(n=5000, seed=42):
    rng = np.random.default_rng(seed)
    seg = rng.choice(["Consumer","Corporate","Small Business"], n, p=[.55,.25,.20])
    tenure = rng.gamma(3, 8, n).round(1)
    disc = np.clip(rng.beta(2, 6, n) + (seg=="Corporate")*0.05, 0, .6)
    freq = np.clip(rng.poisson(4 + tenure/12, n), 0, None)
    spend = np.exp(rng.normal(4.2 + 0.25*(seg=="Corporate") - 0.6*disc + 0.01*tenure, .5, n))
    p = 1/(1+np.exp(-(-1 + 0.05*tenure + 0.15*freq - 2.0*np.maximum(disc-.3, 0)*3 + 0.8*disc)))
    ret = rng.binomial(1, p)
    df = pd.DataFrame({COL["discount"]:disc, COL["retained"]:ret, COL["spend"]:spend,
                       COL["segment"]:seg, COL["tenure"]:tenure, COL["frequency"]:freq})
    idx = rng.choice(n, 25, replace=False)          # planted outliers
    df.loc[idx, COL["spend"]] *= rng.uniform(8, 20, 25)
    return df

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH); SOURCE = DATA_PATH
else:
    df = make_synthetic(); SOURCE = "synthetic demo data"
print("Source:", SOURCE, "| shape:", df.shape)
df.head()

## 1. Data overview

In [ ]:
display(df.dtypes.to_frame("dtype").join(df.isna().sum().rename("missing")).join(df.nunique().rename("unique")))
num_cols = df.select_dtypes("number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Numeric:", num_cols); print("Categorical:", cat_cols)

## 2. Descriptive statistics
Mean, median, standard deviation, quartiles, IQR, skewness and kurtosis for every numeric column.

In [ ]:
desc = df[num_cols].describe().T
desc["median"] = df[num_cols].median()
desc["IQR"] = desc["75%"] - desc["25%"]
desc["skew"] = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurt()
desc["CV"] = desc["std"]/desc["mean"]
desc = desc[["count","mean","median","std","min","25%","50%","75%","max","IQR","skew","kurtosis","CV"]]
desc

In [ ]:
for c in cat_cols:
    print(f"\n--- {c} ---")
    display(df[c].value_counts(dropna=False).to_frame("count").assign(share=lambda d: d["count"]/d["count"].sum()))

## 3. Distributions
Histograms with KDE. Skewed variables are flagged for log-transformation.

In [ ]:
n = len(num_cols); ncol = 3; nrow = int(np.ceil(n/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 3.6*nrow)); axes = np.array(axes).ravel()
for ax, c in zip(axes, num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, color="#3b6ea5")
    ax.axvline(df[c].mean(), color="crimson", ls="--", label="mean")
    ax.axvline(df[c].median(), color="green", ls=":", label="median")
    ax.set_title(f"{c} (skew={df[c].skew():.2f})"); ax.legend()
for ax in axes[n:]: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Anomaly detection: box plots and IQR / z-score rules

In [ ]:
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 3.2*nrow)); axes = np.array(axes).ravel()
for ax, c in zip(axes, num_cols):
    sns.boxplot(x=df[c], ax=ax, color="#9ecae1", flierprops=dict(marker="o", markerfacecolor="crimson", markersize=3))
    ax.set_title(c)
for ax in axes[n:]: ax.axis("off")
plt.tight_layout(); plt.show()

def outlier_table(d, cols):
    rows = []
    for c in cols:
        s = d[c].dropna(); q1, q3 = s.quantile([.25,.75]); iqr = q3-q1
        lo, hi = q1-1.5*iqr, q3+1.5*iqr
        z = np.abs(stats.zscore(s))
        rows.append({"column":c, "IQR_outliers":int(((s<lo)|(s>hi)).sum()),
                     "pct_IQR":100*((s<lo)|(s>hi)).mean(), "z>3":int((z>3).sum()),
                     "lower_fence":lo, "upper_fence":hi})
    return pd.DataFrame(rows).set_index("column")
outliers = outlier_table(df, num_cols); outliers

## 5. Correlation analysis

In [ ]:
pearson = df[num_cols].corr(method="pearson")
spearman = df[num_cols].corr(method="spearman")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
mask = np.triu(np.ones_like(pearson, dtype=bool))
for ax, m, t in zip(axes, [pearson, spearman], ["Pearson (linear)", "Spearman (rank / monotonic)"]):
    sns.heatmap(m, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, square=True)
    ax.set_title(t)
plt.tight_layout(); plt.show()

pairs = (pearson.where(~mask).stack().rename("pearson").to_frame()
         .join(spearman.where(~mask).stack().rename("spearman")))
pairs["abs"] = pairs["pearson"].abs()
pairs.sort_values("abs", ascending=False).drop(columns="abs").head(8)

## 6. Multivariate views

In [ ]:
sample = df.sample(min(len(df), 1500), random_state=1)
hue = COL["segment"] if COL["segment"] in df.columns else None
g = sns.pairplot(sample, vars=[c for c in [COL["discount"], COL["spend"], COL["tenure"], COL["frequency"]] if c in df.columns],
                 hue=hue, corner=True, plot_kws=dict(alpha=.35, s=14))
g.fig.suptitle("Pairwise relationships by segment", y=1.02); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if hue:
    sns.boxplot(data=df, x=hue, y=COL["spend"], ax=axes[0]); axes[0].set_yscale("log"); axes[0].set_title("Spend by segment (log scale)")
    sns.barplot(data=df, x=hue, y=COL["retained"], ax=axes[1], errorbar=("ci",95)); axes[1].set_title("Retention rate by segment (95% CI)")
plt.tight_layout(); plt.show()

## 7. Hypothesis testing
All tests use α = 0.05. For each test we report the statistic, p-value and an **effect size**, because with large samples almost everything is "significant" and practical size matters.

| # | Business hypothesis | H0 | Test |
|---|---|---|---|
| H1 | Customers who receive higher discounts are retained at a different rate | Mean discount is equal for retained vs churned | Welch t-test + Mann-Whitney U, Cohen's d |
| H2 | Average spend differs across customer segments | All segment means are equal | Kruskal-Wallis (+ one-way ANOVA), epsilon-squared |
| H3 | Tenure is associated with order frequency | Spearman rho = 0 | Spearman correlation with bootstrap 95% CI |

In [ ]:
ALPHA = 0.05
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1)+(nb-1)*b.var(ddof=1))/(na+nb-2))
    return (a.mean()-b.mean())/sp

# ---- H1: discount vs retention ----
r = df.loc[df[COL["retained"]]==1, COL["discount"]].dropna()
c = df.loc[df[COL["retained"]]==0, COL["discount"]].dropna()
t, p_t = stats.ttest_ind(r, c, equal_var=False)
u, p_u = stats.mannwhitneyu(r, c, alternative="two-sided")
d = cohens_d(r, c)
h1 = pd.Series({"mean discount (retained)":r.mean(), "mean discount (churned)":c.mean(),
                "Welch t":t, "p (t-test)":p_t, "Mann-Whitney p":p_u, "Cohen's d":d,
                "decision":"Reject H0" if p_t<ALPHA else "Fail to reject H0"})
display(h1.to_frame("H1"))

# Retention rate by discount band
bands = pd.cut(df[COL["discount"]], [0,.1,.2,.3,1], include_lowest=True)
band_tbl = df.groupby(bands)[COL["retained"]].agg(["mean","count"]).rename(columns={"mean":"retention_rate"})
display(band_tbl)
ct = pd.crosstab(bands, df[COL["retained"]]); chi2, p_chi, _, _ = stats.chi2_contingency(ct)
print(f"Chi-square retention vs discount band: chi2={chi2:.2f}, p={p_chi:.3g}")
band_tbl["retention_rate"].plot(kind="bar", color="#3b6ea5", figsize=(6,3.5), title="Retention rate by discount band"); plt.ylabel("rate"); plt.show()

In [ ]:
# ---- H2: spend across segments ----
groups = [g[COL["spend"]].dropna().values for _, g in df.groupby(COL["segment"])]
F, p_f = stats.f_oneway(*groups)
H, p_h = stats.kruskal(*groups)
N, k = sum(len(g) for g in groups), len(groups)
eps2 = (H-k+1)/(N-k)
h2 = pd.Series({"ANOVA F":F, "ANOVA p":p_f, "Kruskal H":H, "Kruskal p":p_h,
                "epsilon-squared":eps2, "decision":"Reject H0" if p_h<ALPHA else "Fail to reject H0"})
display(h2.to_frame("H2"))
display(df.groupby(COL["segment"])[COL["spend"]].agg(["count","mean","median","std"]))

# Pairwise Mann-Whitney with Bonferroni correction
from itertools import combinations
names = sorted(df[COL["segment"]].unique()); m = len(list(combinations(names,2)))
rows = []
for a, b in combinations(names, 2):
    _, p = stats.mannwhitneyu(df.loc[df[COL["segment"]]==a, COL["spend"]], df.loc[df[COL["segment"]]==b, COL["spend"]])
    rows.append({"pair":f"{a} vs {b}", "p_raw":p, "p_bonferroni":min(p*m,1)})
pd.DataFrame(rows).set_index("pair")

In [ ]:
# ---- H3: tenure vs frequency ----
x, y = df[COL["tenure"]], df[COL["frequency"]]
rho, p_rho = stats.spearmanr(x, y)
rng = np.random.default_rng(0); boots = []
for _ in range(1000):
    i = rng.integers(0, len(df), len(df)); boots.append(stats.spearmanr(x.iloc[i], y.iloc[i])[0])
lo, hi = np.percentile(boots, [2.5, 97.5])
h3 = pd.Series({"Spearman rho":rho, "p-value":p_rho, "95% CI low":lo, "95% CI high":hi,
                "decision":"Reject H0" if p_rho<ALPHA else "Fail to reject H0"})
display(h3.to_frame("H3"))
idx = x.sample(min(len(df),1500), random_state=2).index
sns.regplot(x=x.loc[idx], y=y.loc[idx], scatter_kws=dict(alpha=.25, s=12), line_kws=dict(color="crimson"))
plt.title("Tenure vs order frequency"); plt.show()

## 8. Top 5 critical findings

In [ ]:
summary = {
 "H1 discount vs retention": dict(p=p_t, effect=f"d = {d:.2f}"),
 "H2 spend by segment":      dict(p=p_h, effect=f"eps2 = {eps2:.3f}"),
 "H3 tenure vs frequency":   dict(p=p_rho, effect=f"rho = {rho:.2f}"),
}
display(pd.DataFrame(summary).T)
top_corr = pairs.assign(abs=pairs["pearson"].abs()).sort_values("abs", ascending=False).head(1)
print("Strongest linear correlation:", top_corr.index[0], round(top_corr["pearson"].iloc[0], 3))
worst = outliers.sort_values("pct_IQR", ascending=False).head(1)
print("Most outlier-heavy column:", worst.index[0], f"({worst['pct_IQR'].iloc[0]:.1f}% beyond IQR fences)")

### Findings & recommendations
> Replace the bracketed numbers with the values printed above once you run the notebook on your real data. The structure below is the expected commentary.

1. **Distribution shape / skew.** Monetary variables (e.g. spend) are strongly right-skewed (mean > median, high skew and kurtosis). *Recommendation:* report medians, use log-scale or robust models, and avoid mean-only KPIs.
2. **Anomalies.** A small share [x%] of records lie beyond the IQR fences for spend, driven by a handful of extreme orders. *Recommendation:* review for data-entry errors or bulk/B2B orders, then winsorize or segment rather than delete blindly.
3. **Discount and retention (H1).** Retained customers received [higher/lower] average discounts (d = [x], p = [x]); the band table shows retention [rises then plateaus / falls] beyond [x]% discount. *Recommendation:* cap discounts at the point where retention stops improving to protect margin.
4. **Segment differences (H2).** Spend differs significantly across segments (Kruskal p = [x], eps2 = [x]); pairwise tests show [segment A] differs from [segment B]. *Recommendation:* tailor pricing and outreach per segment.
5. **Tenure drives engagement (H3).** Tenure and order frequency are [positively] associated (rho = [x], 95% CI [lo, hi]). *Recommendation:* invest in early-life onboarding, since longer-tenure customers order more often.

**Caveats:** correlation is not causation; discounts may be targeted at at-risk customers (confounding); multiple testing inflates false positives, so effect sizes and Bonferroni-corrected p-values matter more than raw p-values alone.